In [1]:
from scipy.stats import qmc
import numpy as np
import pandas as pd

from scipy.stats import norm
from scipy.optimize import minimize
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel as C
from sklearn.gaussian_process.kernels import Matern

In [2]:
#Function 4
#Extracting updated data and turning it into a pandas dataframe
data = pd.read_csv('Data/Week 6 - Function 4.csv') 
columns = ['Input 1', 'Input 2','Input 3', 'Input 4', 'Outputs']
#Remove columns with NaN values
data = data.dropna(axis = 1)
data.columns = columns
#Add Week 7's data to our pandas dataframe
new_data7 = np.array([0.448159, 0.518735, 0.367877, 0.410965,
                      -1.5194726783305934])
data.loc[len(data)] = new_data7
#Add Week 8's data to our pandas dataframe
new_data8 = np.array([0.404520, 0.368851, 0.432035, 0.470888,
                      -0.6709532563484824])
data.loc[len(data)] = new_data8
#Add Week 9's data to our pandas dataframe
new_data9 = np.array([0.376941, 0.344380, 0.389150, 0.371290,
                      0.3057430767883038])
data.loc[len(data)] = new_data9
#Add Week 10's data to our pandas dataframe
new_data10 = np.array([0.436862, 0.349205, 0.342986, 0.358299,
                      0.36952809492683203])
data.loc[len(data)] = new_data10
#Add Week 11's data to our pandas dataframe
new_data11 = np.array([0.410581, 0.393007, 0.429436, 0.356452,
                      0.5592060627077875])
data.loc[len(data)] = new_data11
data

,Input 1,Input 2,Input 3,Input 4,Outputs
0,0.896981,0.725628,0.175404,0.701694,-22.108290
1,0.889356,0.499588,0.539269,0.508783,-14.601400
2,0.250946,0.033693,0.145380,0.494932,-11.699930
3,0.346962,0.006250,0.760564,0.613024,-16.053770
4,0.124871,0.129770,0.384400,0.287076,-10.069630
5,0.801303,0.500231,0.706645,0.195103,-15.487080
6,0.247708,0.060445,0.042186,0.441324,-12.681690
7,0.746702,0.757092,0.369353,0.206566,-16.026400
8,0.400665,0.072574,0.886768,0.243842,-17.049240
9,0.626071,0.586751,0.438806,0.778858,-12.741770


In [3]:
#Extract the Data Into Numpy Arrays
X = np.array(data[['Input 1', 'Input 2', 'Input 3', 'Input 4']])
Y = np.array(data[['Outputs']])


In [4]:
#Using latin hypercube sampling to initialise a set of candidate points in a trust region
#Initialise points in the unit hypercube
d = 4
n = 40000
sampler = qmc.LatinHypercube(d, seed = 42)
grid = sampler.random(n) 

#Scale points generated in the unit hypercube to be inside trust region
#Trust hypercube of radius delta centered at best point with from observed data is 
#[x_best-delta, x_best_plus]
x_best = X[np.argmax(Y)]
#Set radius of hypercube
delta = 0.2
#lower and upper bounds of hypercube
lb = x_best-delta 
ub = x_best+delta

#scale points from unit hypercube to be inside trust region
scaled_grid = lb + grid * (ub-lb) 


In [5]:
#We now set the Kernel to be the Matern Kernel
nu = 2.5
kernel = Matern(
    length_scale_bounds=(1e-2, 2),
    nu= nu
)
print("Kernel:", kernel)
gp = GaussianProcessRegressor(
    kernel=kernel,
    alpha=1e-10,
    normalize_y = True)
print(gp)
gp.fit(X,Y)

mu, std = gp.predict(scaled_grid, return_std = True)

Kernel: Matern(length_scale=1, nu=2.5)
GaussianProcessRegressor(kernel=Matern(length_scale=1, nu=2.5),
                         normalize_y=True)


In [6]:
#Calculate UCB
beta = 0.2 #set exploration parameter
UCB = mu + beta*std 
#Find point that maximises UCB acquisition function
x_next = scaled_grid[np.argmax(UCB)]
print("Next suggested point:", np.round(x_next,6))


Next suggested point: [0.425993 0.359475 0.408547 0.356329]


In [3]:

#Week 12
#Add new data
new_data = np.array([0.425993, 0.359475, 0.408547, 0.3563291,0.680443696447409])
data.loc[len(data)] = new_data
data

,Input 1,Input 2,Input 3,Input 4,Outputs
0,0.896981,0.725628,0.175404,0.701694,-22.108290
1,0.889356,0.499588,0.539269,0.508783,-14.601400
2,0.250946,0.033693,0.145380,0.494932,-11.699930
3,0.346962,0.006250,0.760564,0.613024,-16.053770
4,0.124871,0.129770,0.384400,0.287076,-10.069630
5,0.801303,0.500231,0.706645,0.195103,-15.487080
6,0.247708,0.060445,0.042186,0.441324,-12.681690
7,0.746702,0.757092,0.369353,0.206566,-16.026400
8,0.400665,0.072574,0.886768,0.243842,-17.049240
9,0.626071,0.586751,0.438806,0.778858,-12.741770


In [6]:
#Extract the Data Into Numpy Arrays
X = np.array(data[['Input 1', 'Input 2', 'Input 3', 'Input 4']])
Y = np.array(data[['Outputs']])

In [11]:
#Using latin hypercube sampling to initialise a set of candidate points in a trust region
#Initialise points in the unit hypercube
d = 4
n = 40000
sampler = qmc.LatinHypercube(d, seed = 42)
grid = sampler.random(n) 

#Scale points generated in the unit hypercube to be inside trust region
#Trust hypercube of radius delta centered at best point with from observed data is 
#[x_best-delta, x_best_plus]
x_best = X[np.argmax(Y)]
#Increase radius of hypercube by 10% since there was an improvement in last weeks function
delta = 0.2*1.1
#lower and upper bounds of hypercube
lb = x_best-delta 
ub = x_best+delta

#scale points from unit hypercube to be inside trust region
scaled_grid = lb + grid * (ub-lb) 
print("Center of hypercube:", x_best)
print("Upper bound", ub)
print("lower bound", lb)

Center of hypercube: [0.425993  0.359475  0.408547  0.3563291]
Upper bound [0.645993  0.579475  0.628547  0.5763291]
lower bound [0.205993  0.139475  0.188547  0.1363291]


In [12]:
#We now set the Kernel to be the Matern Kernel
nu = 2.5
kernel = Matern(
    length_scale_bounds=(1e-2, 2),
    nu= nu
)
print("Kernel:", kernel)
gp = GaussianProcessRegressor(
    kernel=kernel,
    alpha=1e-10,
    normalize_y = True)
print(gp)
gp.fit(X,Y)

mu, std = gp.predict(scaled_grid, return_std = True)

Kernel: Matern(length_scale=1, nu=2.5)
GaussianProcessRegressor(kernel=Matern(length_scale=1, nu=2.5),
                         normalize_y=True)


In [13]:
#Calculate UCB
beta = 0.2 #set exploration parameter
UCB = mu + beta*std 
#Find point that maximises UCB acquisition function
x_next = scaled_grid[np.argmax(UCB)]
print("Next suggested point:", np.round(x_next,6))

Next suggested point: [0.437455 0.388731 0.405269 0.377743]


In [4]:
#Week 13
#Add new data
new_data = np.array([0.437455, 0.388731, 0.405269, 0.377743, 0.17685467072547256])
data.loc[len(data)] = new_data
data

,Input 1,Input 2,Input 3,Input 4,Outputs
0,0.896981,0.725628,0.175404,0.701694,-22.108290
1,0.889356,0.499588,0.539269,0.508783,-14.601400
2,0.250946,0.033693,0.145380,0.494932,-11.699930
3,0.346962,0.006250,0.760564,0.613024,-16.053770
4,0.124871,0.129770,0.384400,0.287076,-10.069630
5,0.801303,0.500231,0.706645,0.195103,-15.487080
6,0.247708,0.060445,0.042186,0.441324,-12.681690
7,0.746702,0.757092,0.369353,0.206566,-16.026400
8,0.400665,0.072574,0.886768,0.243842,-17.049240
9,0.626071,0.586751,0.438806,0.778858,-12.741770


In [5]:
#Extract the Data Into Numpy Arrays
X = np.array(data[['Input 1', 'Input 2', 'Input 3', 'Input 4']])
Y = np.array(data[['Outputs']])

In [7]:
#Using latin hypercube sampling to initialise a set of candidate points in a trust region
#Initialise points in the unit hypercube
d = 4
n = 40000
sampler = qmc.LatinHypercube(d, seed = 42)
grid = sampler.random(n) 

#Scale points generated in the unit hypercube to be inside trust region
#Trust hypercube of radius delta centered at best point with from observed data is 
#[x_best-delta, x_best_plus]
x_best = X[np.argmax(Y)]
#Decrease radius of hypercube by 10% since there was no improvement in last weeks function
delta = 0.2*1.1*0.9
#lower and upper bounds of hypercube
lb = x_best-delta 
ub = x_best+delta

#scale points from unit hypercube to be inside trust region
scaled_grid = lb + grid * (ub-lb) 
print("Center of hypercube:", x_best)
print("Upper bound", ub)
print("lower bound", lb)

Center of hypercube: [0.425993  0.359475  0.408547  0.3563291]
Upper bound [0.623993  0.557475  0.606547  0.5543291]
lower bound [0.227993  0.161475  0.210547  0.1583291]


In [8]:
#We now set the Kernel to be the Matern Kernel
nu = 2.5
kernel = Matern(
    length_scale_bounds=(1e-2, 2),
    nu= nu
)
print("Kernel:", kernel)
gp = GaussianProcessRegressor(
    kernel=kernel,
    alpha=1e-10,
    normalize_y = True)
print(gp)
gp.fit(X,Y)

mu, std = gp.predict(scaled_grid, return_std = True)

Kernel: Matern(length_scale=1, nu=2.5)
GaussianProcessRegressor(kernel=Matern(length_scale=1, nu=2.5),
                         normalize_y=True)


In [11]:
#Calculate UCB
beta = 0.2 #set exploration parameter
UCB = mu + beta*std 
#Find point that maximises UCB acquisition function
x_next = scaled_grid[np.argmax(UCB)]
print("Next suggested point:", np.round(x_next,6))

Next suggested point: [0.408891 0.342949 0.459077 0.325853]
